# H&M Colab Training

Run the existing recommendation implementation through verified CLI stages.
This release supports CPU smoke and an explicitly enabled pilot backend.
Pilot uses global legal statistics, the full catalog, a bounded learned-model
cohort, and F1/Validation query rosters. Full, final Test and deployment remain
disabled. New-release Colab/CUDA/pilot execution is not yet verified.

## Contents

1. [Objective and execution scope](#1-objective-and-execution-scope)
2. [Configuration and Drive mount](#2-configuration-and-drive-mount)
3. [Release verification and staging](#3-release-verification-and-staging)
4. [Environment and resource preflight](#4-environment-and-resource-preflight)
5. [Input staging and validation](#5-input-staging-and-validation)
6. [Smoke and device verification](#6-smoke-and-device-verification)
7. [Pilot or full gates](#7-pilot-or-full-gates)
8. [Validation and reload](#8-validation-and-reload)
9. [Persistence and results](#9-persistence-and-results)

## 1. Objective and execution scope

Preserve all six retrieval channels (60/30/30/50/50/50), the formal disjoint
time windows, complete ground truth, native candidates, and one LambdaMART.
Smoke exercises four training folds; pilot deliberately executes F1 and Validation.
Models, features, metrics and resource guards live in `src/hm_recsys/`.
CPU smoke is a bounded wiring check, not a benchmark. Existing source embeddings
and indexes are read-only; only learned model indexes are generated.

Upload this notebook independently. Upload the ZIP, matching `.sha256`, and
`.manifest.json` to the Drive project's `releases/` folder. The user supplied successful real Colab CPU smoke evidence for the prior
release. This is not CUDA or real-pilot evidence for this new release.

## 2. Configuration and Drive mount

Replace `RELEASE_NAME` with the actual built archive name. Use a new `RUN_ID`
for a new experiment; use `RUN_ACTION = "resume"` and the original ID for
recovery. Cell reruns validate existing stages. They never reset a run.
A stale writer lock requires checking that the previous runtime has stopped
before explicitly setting `RECOVER_LOCK = True`.

For an intentional pilot, use a new run ID and the resolved release name from
the packaged handoff. Change these values before running any stage:

```python
RUN_PROFILE = "pilot"
RUN_ID = "colab_pilot_v1"
RUN_ACTION = "run"
ALLOW_PILOT_RUN = True
PILOT_DEVICE = "cuda"
RUN_CUDA_PROBE = True
ALLOW_FULL_RUN = False
RUN_FINAL_TEST = False
```

CPU can be selected explicitly with `PILOT_DEVICE = "cpu"` and
`RUN_CUDA_PROBE = False`. CUDA requests never fall back to CPU. These are
resource-limited experiments, not a complete Validation benchmark.

In [ ]:
from pathlib import Path

DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/projects/e_commerce_recsys")
PROJECT_ROOT = Path("/content/e_commerce_recsys")
SOURCE_DATA_ROOT = DRIVE_PROJECT_ROOT / "data"
LOCAL_DATA_ROOT = PROJECT_ROOT / "data"
RELEASE_NAME = "hm_recsys_<build_id>.zip"
RUN_PROFILE = "smoke"
RUN_ID = "colab_smoke_v1"
RUN_ACTION = "run"
ALLOW_PILOT_RUN = False
ALLOW_FULL_RUN = False
PILOT_DEVICE = "cuda"
RUN_FINAL_TEST = False
RUN_CUDA_PROBE = False
RECOVER_LOCK = False

if "<" in RELEASE_NAME or not RELEASE_NAME.endswith(".zip"):
    raise ValueError("Set RELEASE_NAME to the built ZIP filename before running")
if RUN_PROFILE not in ("smoke", "pilot"):
    raise ValueError("Full execution is disabled for this release")
if RUN_PROFILE == "pilot" and not ALLOW_PILOT_RUN:
    raise ValueError("Pilot requires explicit ALLOW_PILOT_RUN = True")
if RUN_ACTION not in ("run", "resume") or RUN_FINAL_TEST:
    raise ValueError("Unsupported action or final Test request")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Release verification and staging

Verify the archive hash against both sidecars before executing its standard-library
bootstrap. That bootstrap validates every member and rejects traversal, links,
data files and incompatible existing checkouts. Keep all three files from the
same trusted build; checksums detect corruption, not an untrusted publisher.

In [ ]:
import hashlib
import json
import zipfile

release_path = DRIVE_PROJECT_ROOT / "releases" / RELEASE_NAME
release_manifest = json.loads(release_path.with_suffix(".manifest.json").read_text())
checksum = release_path.with_suffix(".sha256").read_text().split()[0]
with release_path.open("rb") as handle:
    actual_sha = hashlib.file_digest(handle, "sha256").hexdigest()
if actual_sha != checksum or actual_sha != release_manifest["archive_sha256"]:
    raise ValueError("Release checksum mismatch")
with zipfile.ZipFile(release_path) as archive:
    bootstrap = archive.read("src/hm_recsys/colab/bootstrap.py")
expected = release_manifest["files"]["src/hm_recsys/colab/bootstrap.py"]
if hashlib.sha256(bootstrap).hexdigest() != expected["sha256"]:
    raise ValueError("Bootstrap checksum mismatch")
namespace = {"__name__": "release_bootstrap"}
exec(compile(bootstrap, "release_bootstrap.py", "exec"), namespace)
namespace["unpack"](release_path, release_manifest, PROJECT_ROOT)
print("Verified build:", release_manifest["build_id"])

## 4. Environment and resource preflight

Install only the pinned uv launcher into a separate tool directory, then create
the release's locked Python 3.12 environment. This does not modify the notebook
kernel's CUDA-enabled PyTorch. Linux wheel and CUDA compatibility still require
actual execution; stop on installation/probe failure rather than replacing models
or regenerating the lock. Reserve at least 15 GiB for environment installation
plus 10 GiB free disk; input and model storage are additional requirements.

In [ ]:
import shutil
import sys

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from hm_recsys.colab.notebook import command

LOG_ROOT = LOCAL_DATA_ROOT / "runs/colab" / RUN_ID / "notebook_logs"
if shutil.disk_usage(PROJECT_ROOT).free < 25 * 1024**3:
    raise RuntimeError("Insufficient disk reserve for environment preparation")
TOOLS = Path("/content/hm_colab_tools")
command(
    [sys.executable, "-m", "pip", "install", "--target", str(TOOLS), "uv==0.13.6"],
    PROJECT_ROOT,
    LOG_ROOT / "environment.log",
)
UV = str(TOOLS / "bin/uv")
command([UV, "python", "install", "3.12"], PROJECT_ROOT, LOG_ROOT / "environment.log")
command(
    [UV, "sync", "--locked", "--python", "3.12"],
    PROJECT_ROOT,
    LOG_ROOT / "environment.log",
)
PYTHON = str(PROJECT_ROOT / ".venv/bin/python")
PERSISTENT_ROOT = SOURCE_DATA_ROOT / "runs/colab"
PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)
CLI = [PYTHON, str(PROJECT_ROOT / "scripts/run_pipeline.py")]
OPTIONS = [
    "--profile",
    RUN_PROFILE,
    "--run-id",
    RUN_ID,
    "--data-root",
    str(LOCAL_DATA_ROOT),
    "--persistent-root",
    str(PERSISTENT_ROOT),
    "--mount-root",
    "/content/drive",
]
if RUN_PROFILE == "pilot":
    OPTIONS += ["--device", PILOT_DEVICE]
    if ALLOW_PILOT_RUN:
        OPTIONS += ["--allow-pilot"]
if RECOVER_LOCK:
    OPTIONS += ["--recover-lock"]
print(
    {
        "profile": RUN_PROFILE,
        "run_id": RUN_ID,
        "action": RUN_ACTION,
        "allow_full": ALLOW_FULL_RUN,
        "run_final_test": RUN_FINAL_TEST,
    }
)
command(CLI + ["plan"] + OPTIONS, PROJECT_ROOT, LOG_ROOT / "plan.log")
print((LOG_ROOT / "plan.log").read_text()[-12000:])

## 5. Input staging and validation

Copy only the 12 required files in `original data/`, `text_features/` and
`image_features/`. Interrupted copies are not committed. Reuse requires local
checksums and source size/mtime agreement; add `--reverify` to rehash source bytes.
No old run or feature output is an input. All training happens on local `/content`.
Preflight checks schemas, mapping contracts, date splits and bounded vector
samples; it is not a full-array numerical audit.

Pilot stages build disk-backed canonical history and complete global cutoff
statistics locally. The 5,000-user gradient cohort does not restrict popularity,
ItemCF, active items or truth. Only F1/Validation query rosters are capped at
1,000 each; the full supplied article catalog is retained.

In [ ]:
command(
    CLI
    + [
        "stage-inputs",
        "--source-root",
        str(SOURCE_DATA_ROOT),
        "--data-root",
        str(LOCAL_DATA_ROOT),
    ],
    PROJECT_ROOT,
    LOG_ROOT / "staging.log",
)
command(CLI + ["preflight"] + OPTIONS, PROJECT_ROOT, LOG_ROOT / "preflight.log")

## 6. Smoke and device verification

The following command executes the selected profile through its learned index.
Pilot also has a required isolated device-probe stage before fitting. Formal
Two-Tower training and encoding use the recorded profile device, independently
of the optional notebook probe. An unavailable requested CUDA device fails.

The optional probe below uses a new output ID on every invocation, preserving
all earlier evidence. It performs an actual shared-model update and reload.
Its success is separate from training completion and from fresh-runtime recovery.
Models, FAISS indexing and LambdaMART remain isolated in separate processes.

In [ ]:
import uuid

if RUN_CUDA_PROBE:
    probe_output = (
        LOCAL_DATA_ROOT
        / "runs"
        / "colab"
        / f"{RUN_ID}_cuda_probe_{uuid.uuid4().hex[:12]}"
    )
    command(
        [
            PYTHON,
            str(PROJECT_ROOT / "scripts" / "probe_device.py"),
            "--device",
            "cuda",
            "--output",
            str(probe_output),
        ],
        PROJECT_ROOT,
        LOG_ROOT / "cuda_probe.log",
    )
command(
    CLI + [RUN_ACTION, "--until", "index_towers"] + OPTIONS,
    PROJECT_ROOT,
    LOG_ROOT / "smoke_base.log",
)

## 7. Pilot or full gates

Pilot requires `RUN_PROFILE = "pilot"` and `ALLOW_PILOT_RUN = True`; it does
not require `ALLOW_FULL_RUN`. Its base-training cohort is at most 5,000 legal
past users; all global statistics and eligible catalog items remain global.
WALS runs at most 3 iterations with 64 factors. Two-Tower uses batch 512,
at most 20,000 targets, 1 epoch and 100 updates. LambdaMART has at most 100 rounds.
Actual counts are saved. No cap is a promise of memory feasibility.

The shared CLI chooses F1 and Validation for pilot, rather than looking for
missing F2–F4 artifacts. Full stays blocked even if its opt-in is changed.
Resource failures stop the run; no hidden sampling or algorithm replacement
is permitted. Use the pilot runbook for explicit commands and resume steps.

In [ ]:
print(
    {
        "selected_profile": RUN_PROFILE,
        "pilot_opt_in": ALLOW_PILOT_RUN,
        "full_enabled": False,
        "final_test_enabled": RUN_FINAL_TEST,
    }
)
print("The next cell continues this run through its declared folds and reload.")

## 8. Validation and reload

Continue the selected profile with frozen base weights, dynamic cutoff histories,
native features and LambdaMART. Smoke uses four training folds; pilot uses F1.
Validation retains full truth and zero-hit/zero-candidate queries within its
explicit roster. Train-only negative sampling never affects Validation.

Fresh-process encoding, feature reconstruction and scoring must match saved
candidate identities, features, scores and Top-12. No future labels are read
by reload inference. These are pilot diagnostics, not full Validation results.

In [ ]:
command(
    CLI + [RUN_ACTION, "--until", "reload"] + OPTIONS,
    PROJECT_ROOT,
    LOG_ROOT / "validation_reload.log",
)

## 9. Persistence and results

Every successful stage publishes verified output deltas, followed by its
completion manifest, to Drive. Completion is never inferred from directory
existence. Results and necessary reload inputs live below
`data/runs/colab/<run_id>/`; source inputs stay untouched.

In a new runtime, reopen this notebook, use the same release, profile and run ID,
set `RUN_ACTION = "resume"`, then run from the top. Local inputs are restaged,
completed artifacts are verified/restored, and incomplete stages continue or
restart at the documented boundary. Changed code/config/input/schema/cutoff or
numerical runtime is rejected. Consult `docs/archi/colab_runbook.md` in the ZIP.

No new-release Colab execution results are embedded in this notebook. Stop after the selected smoke or pilot Validation/reload milestone;
full and final Test remain disabled. Pilot publishes a small `pilot_review_<identity>.zip`
with aggregate diagnostics and identities, excluding raw customer IDs, histories
and feature tables. Fresh-runtime recovery on real Drive still needs verification.

In [ ]:
command(CLI + ["status"] + OPTIONS, PROJECT_ROOT, LOG_ROOT / "status.log")
run_root = LOCAL_DATA_ROOT / "runs/colab" / RUN_ID
status = json.loads((run_root / "last_command_result.json").read_text())
if not status["complete"]:
    raise RuntimeError("Some stages or persistent artifacts are incomplete")
locations = json.loads((run_root / "artifacts.json").read_text())
summary = json.loads((run_root / "run_summary.json").read_text())
print(
    {
        "status": summary["status"],
        "validation": summary["validation"],
        "ranker_rounds": summary["ranker"]["actual_rounds"],
        "tower_steps": summary["twotower"]["steps"],
        "reload": summary["reload"]["status"],
    }
)
print("Verified persistent run:", PERSISTENT_ROOT / RUN_ID)
if RUN_PROFILE == "pilot":
    review = json.loads((run_root / "review_manifest.json").read_text())
    print("Review bundle:", PERSISTENT_ROOT / RUN_ID / review["archive"])